# 02 — Balanced Transformer Preprocessing

This notebook creates the only processed dataset used by the next modelling stages. It removes exact duplicates, samples the same number of reviews from every rating, applies light BERT-family preprocessing, creates a stratified train/validation split, and saves reproducibility metadata. The raw and official test datasets are never modified.

## Experiment configuration

To scale the experiment later, change only `SAMPLES_PER_CLASS` (for example from `10_000` to `50_000`) and run all cells again. A separate output folder will be created automatically.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.cleaner import preprocess_for_transformer

SAMPLES_PER_CLASS = 10_000  # Change only this value to scale the dataset.
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
TARGET_COLUMN = 'overall'
TEXT_COLUMN = 'reviewText'

INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train_data.csv'
PRODUCT_METADATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'title_brand.csv'
OUTPUT_DIR = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / f'bert_balanced_{SAMPLES_PER_CLASS}_per_class'
)
OUTPUT_DIR

## Load raw training data

In [ ]:
raw_df = pd.read_csv(INPUT_PATH, low_memory=False)
required_columns = {TARGET_COLUMN, TEXT_COLUMN}
missing_columns = required_columns.difference(raw_df.columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'

print('Raw shape:', raw_df.shape)
display(raw_df[TARGET_COLUMN].value_counts().sort_index().to_frame('count'))

## Remove exact duplicates

Exact duplicate rows are removed before sampling so the same observation cannot leak into both train and validation. Duplicate review text with genuinely different records is retained.

In [ ]:
raw_rows = len(raw_df)
deduplicated_df = raw_df.drop_duplicates().reset_index(drop=True)
exact_duplicates_removed = raw_rows - len(deduplicated_df)

print('Exact duplicates removed:', exact_duplicates_removed)
print('Eligible rows:', len(deduplicated_df))
display(deduplicated_df[TARGET_COLUMN].value_counts().sort_index().to_frame('available'))

## Create the reproducible balanced subset

In [ ]:
available_per_class = deduplicated_df[TARGET_COLUMN].value_counts().sort_index()
insufficient = available_per_class[available_per_class < SAMPLES_PER_CLASS]
if not insufficient.empty:
    raise ValueError(
        f'Not enough rows for sampling without replacement: {insufficient.to_dict()}'
    )

balanced_df = (
    deduplicated_df.groupby(TARGET_COLUMN, group_keys=False, sort=True)
    .sample(n=SAMPLES_PER_CLASS, replace=False, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

display(balanced_df[TARGET_COLUMN].value_counts().sort_index().to_frame('sampled'))
print('Balanced shape:', balanced_df.shape)

## Apply light preprocessing and build the selected model input

Applied to review and summary: HTML removal, URL removal, whitespace normalization, and safe handling of non-string values.

Intentionally preserved: case, punctuation, stopwords, negation, numbers, emoji, and natural sentence structure.

A controlled ablation on the same train/validation split selected `summary + verified status + helpful-vote bucket + review` as the strongest input. Product title/brand and style/year were tested but reduced validation micro-F1, so they are retained as separate columns rather than injected into the 128-token model input. Tokenization and truncation belong to the future deep-learning notebook.

In [ ]:
original_review = balanced_df[TEXT_COLUMN].fillna('').astype(str)
original_summary = balanced_df['summary'].fillna('').astype(str)
balanced_df[TEXT_COLUMN] = original_review.map(preprocess_for_transformer)
balanced_df['summary'] = original_summary.map(preprocess_for_transformer)
changed_review_rows = int(original_review.ne(balanced_df[TEXT_COLUMN]).sum())
changed_summary_rows = int(original_summary.ne(balanced_df['summary']).sum())
empty_text_rows = int(balanced_df[TEXT_COLUMN].eq('').sum())

product_df = pd.read_csv(PRODUCT_METADATA_PATH, low_memory=False)
product_df['_completeness'] = product_df[['title', 'brand']].notna().sum(axis=1)
product_df = (
    product_df.sort_values('_completeness', ascending=False)
    .drop_duplicates('asin', keep='first')
    [['asin', 'title', 'brand']]
)
balanced_df = balanced_df.merge(
    product_df, on='asin', how='left', validate='many_to_one'
)
balanced_df['title'] = balanced_df['title'].fillna('').astype(str).map(preprocess_for_transformer)
balanced_df['brand'] = balanced_df['brand'].fillna('').astype(str).map(preprocess_for_transformer)

balanced_df['verified_str'] = (
    balanced_df['verified'].map({True: 'yes', False: 'no'}).fillna('unknown')
)
numeric_vote = pd.to_numeric(
    balanced_df['vote'].astype('string').str.replace(',', '', regex=False),
    errors='coerce',
)
balanced_df['vote_bucket'] = pd.cut(
    numeric_vote.fillna(-1),
    bins=[-2, -0.5, 4.5, 9.5, 49.5, np.inf],
    labels=['missing', '2_to_4', '5_to_9', '10_to_49', '50_plus'],
).astype('string')

balanced_df['model_input'] = (
    'Summary: ' + balanced_df['summary']
    + ' | Verified: ' + balanced_df['verified_str']
    + ' | Helpful votes: ' + balanced_df['vote_bucket'].fillna('missing')
    + ' | Review: ' + balanced_df[TEXT_COLUMN]
)

metadata_coverage = balanced_df['title'].ne('').mean()
print('Changed review rows:', changed_review_rows)
print('Changed summary rows:', changed_summary_rows)
print('Empty review rows after preprocessing:', empty_text_rows)
print(f'Product-title coverage: {metadata_coverage:.2%}')
display(balanced_df[[TARGET_COLUMN, 'summary', 'verified_str', 'vote_bucket', 'reviewText', 'model_input']].head(3))

## Validate and create a stratified train/validation split

In [ ]:
expected_classes = [1, 2, 3, 4, 5]
class_counts = balanced_df[TARGET_COLUMN].value_counts().sort_index()

assert class_counts.index.tolist() == expected_classes
assert class_counts.eq(SAMPLES_PER_CLASS).all()
assert len(balanced_df) == len(expected_classes) * SAMPLES_PER_CLASS
assert not balanced_df.duplicated().any()
assert empty_text_rows == 0
assert balanced_df['model_input'].notna().all()
assert balanced_df['model_input'].str.strip().ne('').all()
assert metadata_coverage > 0.99

train_df, validation_df = train_test_split(
    balanced_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=balanced_df[TARGET_COLUMN],
)
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

split_summary = pd.concat(
    {
        'full': balanced_df[TARGET_COLUMN].value_counts().sort_index(),
        'train': train_df[TARGET_COLUMN].value_counts().sort_index(),
        'validation': validation_df[TARGET_COLUMN].value_counts().sort_index(),
    },
    axis=1,
)
display(split_summary)
print('Validation: PASSED')

## Save isolated processed files and metadata

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
balanced_path = OUTPUT_DIR / 'balanced_reviews.csv'
train_path = OUTPUT_DIR / 'train.csv'
validation_path = OUTPUT_DIR / 'validation.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'

balanced_df.to_csv(balanced_path, index=False)
train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_file': str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    'samples_per_class': SAMPLES_PER_CLASS,
    'random_state': RANDOM_STATE,
    'validation_size': VALIDATION_SIZE,
    'raw_rows': raw_rows,
    'exact_duplicates_removed': exact_duplicates_removed,
    'balanced_rows': len(balanced_df),
    'train_rows': len(train_df),
    'validation_rows': len(validation_df),
    'class_distribution': {str(k): int(v) for k, v in class_counts.items()},
    'changed_review_rows': changed_review_rows,
    'changed_summary_rows': changed_summary_rows,
    'product_title_coverage': metadata_coverage,
    'preprocessing': ['remove_html', 'remove_urls', 'normalize_whitespace'],
    'model_input_column': 'model_input',
    'model_input_fields': ['summary', 'verified_str', 'vote_bucket', 'reviewText'],
    'metadata_considered_but_excluded_from_model_input': [
        'title', 'brand', 'style', 'reviewTime', 'reviewerID',
        'reviewerName', 'asin', 'unixReviewTime'
    ],
    'preserved_for_transformers': [
        'case', 'punctuation', 'stopwords', 'negation', 'numbers', 'emoji'
    ],
    'columns': balanced_df.columns.tolist(),
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print('Saved:')
for path in [balanced_path, train_path, validation_path, metadata_path]:
    print('-', path)

## Output contract

The generated folder is isolated by experiment size. With the current setting it is `data/processed/bert_balanced_10000_per_class/`. Future machine-learning and deep-learning notebooks must use `train.csv` and `validation.csv`; the official raw test set remains untouched until final inference.